# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiknaxTheGreek/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task

The project is a **content-performance prioritisation POC**.

The primary task is **ranking / scoring**: deciding which content pages should be reviewed first.

Three standard ML tasks support the ranking:

1. **Clustering** — group similar pages into performance archetypes.
2. **Classification** — predict whether a page will experience a future decline.
3. **Regression** — predict the size of that decline.
4. **Ranking / scoring** — combine the results to prioritise pages for human review.

Signal analysis and peer-relative comparisons are supporting feature-analysis and feature-engineering steps.

The workflow is:

**observed features → clustering → classification + regression → ranking / scoring**

The final output is a ranked human-review queue.

In [1]:
import pandas as pd
from pathlib import Path

candidate_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]

data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Unique content items: {df['content_id'].nunique():,}")
print(f"Pseudonymized clients: {df['client_id'].nunique():,}")

assert df['content_id'].nunique() == len(df), (
    "Expected one row per pseudonymized content item."
)

methodology = [
    "signal analysis",
    "clustering / peer context",
    "archetype-relative deviations",
    "future-state prediction",
    "impact estimation",
    "ranking / scoring",
    "action hypothesis",
]
print("Methodology:", " -> ".join(methodology))


Rows: 30,000
Columns: 44
Unique content items: 30,000
Pseudonymized clients: 32
Methodology: signal analysis -> clustering / peer context -> archetype-relative deviations -> future-state prediction -> impact estimation -> ranking / scoring -> action hypothesis


## 2. Target or proxy

### Clustering

Clustering is unsupervised, so it has **no target variable**.

It groups similar pages into archetypes that can also be used as peer groups.

### Classification

The classification target is an observed future outcome such as:

`future_decline_30d`

where:

- `1` = future decline;
- `0` = no future decline.

### Regression

The regression target is the observed size of a future decline, such as:

`future_decline_magnitude_30d`

Classification therefore answers:

**Will the page decline?**

Regression answers:

**How large is the decline likely to be?**

### Ranking / scoring

The final ranking does not use a manually created priority label.

It combines information such as:

**decline probability + expected decline size + page exposure/value**

to rank pages for review.

### Time structure

The final targets must use separate past and future periods:

**past features → decision point → future outcome**

The exact windows and target definitions will be fixed in the data-contract stage.

### Starter-data proxy

The starter dataset is only a trailing-90-day snapshot, so it cannot provide a true future target.

For Assignment 3 only:

`decline_proxy = 1` when `trend_direction == "down"`

This is a temporary framing proxy, not a true future label.

Because `trend_direction` is derived from `trend_pct`, neither field may be used as a predictive feature when this proxy is used.

In [2]:
# Transparent starter-data proxy for Assignment 3 framing.
required_proxy_columns = {
    "content_id",
    "trend_direction",
    "trend_pct",
    "impressions_prev_30d",
    "impressions_last_30d",
}
missing = sorted(required_proxy_columns.difference(df.columns))
assert not missing, f"Missing required proxy columns: {missing}"

proxy_frame = df[[
    "content_id",
    "impressions_prev_30d",
    "impressions_last_30d",
    "trend_pct",
    "trend_direction",
]].copy()

proxy_frame["decline_proxy"] = (
    proxy_frame["trend_direction"].str.lower().eq("down").astype("int8")
)

print("Starter proxy: decline_proxy = 1 when trend_direction == 'down'.")
print("This is a current-window proxy, not a future causal or intervention label.\n")
print(proxy_frame["decline_proxy"].value_counts().sort_index())
print(f"Proxy positive rate: {proxy_frame['decline_proxy'].mean():.1%}\n")

display(proxy_frame.head(10))

print("Excluded from predictive features for this proxy: ['trend_direction', 'trend_pct']")

# Later warehouse target structure (not fabricated from this snapshot):
target_schema = pd.DataFrame({
    "field": [
        "future_decline_30d",
        "future_change_magnitude",
    ],
    "role": [
        "future-state classification target",
        "future-movement magnitude outcome",
    ],
    "available_in_starter_snapshot": [False, False],
})
display(target_schema)


Starter proxy: decline_proxy = 1 when trend_direction == 'down'.
This is a current-window proxy, not a future causal or intervention label.

decline_proxy
0    13738
1    16262
Name: count, dtype: int64
Proxy positive rate: 54.2%



,content_id,impressions_prev_30d,impressions_last_30d,trend_pct,trend_direction,decline_proxy
0,content_304f48230142,987,578,-41.4,down,1
1,content_a1fb4e703a9e,5915,2501,-57.7,down,1
2,content_9aa793d4d895,6089,2382,-60.9,down,1
3,content_331d6c4de07b,4206,3626,-13.8,stable,0
4,content_d99b7a2d90ca,6452,4211,-34.7,down,1
5,content_d4084a4bc775,1009,617,-38.9,down,1
6,content_9a34b442b552,13,1,-92.3,down,1
7,content_a63219c6e95a,632,636,0.6,stable,0
8,content_5e6c160719bc,13828,5696,-58.8,down,1
9,content_c27558df2b0c,356,252,-29.2,down,1


Excluded from predictive features for this proxy: ['trend_direction', 'trend_pct']


,field,role,available_in_starter_snapshot
0,future_decline_30d,future-state classification target,false
1,future_change_magnitude,future-movement magnitude outcome,false


## 3. Success metrics

Each ML task has one primary metric and three supporting metrics.

| Task | Primary metric | Secondary metrics |
|---|---|---|
| **Clustering** | **Prediction Strength** | Silhouette Score, Calinski-Harabasz Score, Davies-Bouldin Score |
| **Classification** | **ROC-AUC** | Precision, Recall, F1 Score |
| **Regression** | **RMSE** | MAE, Median Absolute Error, R² |
| **Ranking / scoring** | **Precision@50** | Recall@50, Lift@50, NDCG@50 |

### Clustering

**Prediction Strength** is the primary metric because the archetypes should reproduce on unseen data.

The secondary metrics check cluster separation and compactness.

### Classification

**ROC-AUC** is the primary metric because it measures how well the model separates future declines from non-declines without requiring a fixed classification threshold.

Precision, Recall and F1 are reported to show the practical balance between false alarms and missed declines.

### Regression

**RMSE** is the primary metric because large errors in predicted decline size should be penalised more strongly.

The model must beat a simple baseline that predicts the training-set mean decline magnitude.

MAE, Median Absolute Error and R² provide additional views of prediction error and explained variation.

### Ranking / scoring

**Precision@50** is the primary project metric because the final output is a limited human-review queue.

It measures:

**Of the top 50 recommended pages, how many are genuinely relevant future cases?**

Recall@50 measures coverage, Lift@50 compares the queue with the overall base rate, and NDCG@50 checks whether the most important cases appear near the top.

The learned ranking must be compared with the frozen rule-based baseline using the same pages, future outcomes and `K = 50`.

`K = 50` is the fixed reporting depth for this POC. A real operational review capacity can replace it later if one is provided.

### Supporting analysis

Permutation importance may be used to inspect useful features.

Peer-relative Lift@10% may be used to check whether unusually weak pages within their peer groups are enriched for future declines.

These are supporting analyses, not separate ML tasks.

In [ ]:
# Metric contract for the four standard ML tasks.
import pandas as pd

metric_contract = pd.DataFrame([
    ("Clustering", "Prediction Strength", "Silhouette Score; Calinski-Harabasz Score; Davies-Bouldin Score"),
    ("Classification", "ROC-AUC", "Precision; Recall; F1 Score"),
    ("Regression", "RMSE", "MAE; Median Absolute Error; R²"),
    ("Ranking / scoring", "Precision@50", "Recall@50; Lift@50; NDCG@50"),
], columns=["task", "primary_metric", "secondary_metrics"])

display(metric_contract)
print("Primary project metric: Precision@50")
print("Ranking comparison: learned queue vs frozen rule baseline on the same future outcomes.")
print("K = 50 is the fixed reporting depth for this POC.")


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.